# Proyecto II — Análisis Exploratorio de Datos (EDA)
## Copa Mundial de la FIFA

Este notebook realiza el **análisis exploratorio** de los partidos históricos de la
Copa Mundial de la FIFA. El objetivo es ingerir los datos crudos, limpiarlos, generar
las columnas derivadas necesarias y describir el comportamiento estadístico de los goles.

**Fuente de datos:** repositorio público `martj42/international_results`, filtrado
únicamente por el torneo `FIFA World Cup` (1930 en adelante).

**Flujo del notebook:**
1. Ingesta y filtrado de los datos crudos.
2. Limpieza de tipos y generación de columnas derivadas.
3. Resumen descriptivo.
4. Matriz de correlación.
5. Forma de la distribución de goles (asimetría y curtosis).
6. Detección de partidos atípicos mediante la regla del IQR.
7. Ranking de las mayores goleadas históricas.

In [1]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
from src.ingesta.cargador import CargadorDatos
from src.eda.procesador import ProcesadorEDA

## 1. Ingesta de datos

La clase `CargadorDatos` descarga el CSV completo desde la fuente oficial, conserva
solo los partidos de la Copa Mundial y valida que existan las columnas mínimas
necesarias antes de guardar el archivo en `data/raw/`.

In [2]:
cargador = CargadorDatos()
df_raw = cargador.descargar_y_filtrar_raw()

print(f"Partidos de Mundial cargados: {len(df_raw)}")
display(df_raw.head())

Descargando datos desde el repositorio de GitHub...


Filtrando partidos únicamente de la Copa Mundial de la FIFA...
Archivo raw guardado exitosamente en: ../data/raw/partidos-mundial.csv
Partidos de Mundial cargados: 1068


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
1490,1930-07-13,Belgium,United States,0,3,FIFA World Cup,Montevideo,Uruguay,True
1491,1930-07-13,France,Mexico,4,1,FIFA World Cup,Montevideo,Uruguay,True
1492,1930-07-14,Brazil,Yugoslavia,1,2,FIFA World Cup,Montevideo,Uruguay,True
1493,1930-07-14,Peru,Romania,1,3,FIFA World Cup,Montevideo,Uruguay,True
1494,1930-07-15,Argentina,France,1,0,FIFA World Cup,Montevideo,Uruguay,True


## 2. Limpieza y columnas derivadas

`ProcesadorEDA` convierte los tipos de datos (fechas y marcadores), maneja los valores
nulos y crea las columnas derivadas obligatorias del proyecto:

- **`anio`**: año en que se disputó el partido.
- **`total_goles`**: suma de goles de local y visitante.
- **`diferencia_goles`**: goles del local menos goles del visitante.
- **`ganador`**: `Local`, `Visitante` o `Empate` según el marcador.

El resultado se guarda en `data/processed/partidos-limpios.csv` para ser reutilizado
por el notebook de visualización.

In [3]:
procesador = ProcesadorEDA(df_raw)
procesador.limpieza_datos()
df_procesado = procesador.generar_columnas_derivadas()
cargador.guardar_procesado(df_procesado)

display(df_procesado[['date', 'home_team', 'away_team', 'total_goles',
                      'diferencia_goles', 'ganador']].head())

Limpieza de datos completada.
Archivo procesado guardado exitosamente en: ../data/processed/partidos-limpios.csv


,date,home_team,away_team,total_goles,diferencia_goles,ganador
1490,1930-07-13,Belgium,United States,3,-3,Visitante
1491,1930-07-13,France,Mexico,5,3,Local
1492,1930-07-14,Brazil,Yugoslavia,3,-1,Visitante
1493,1930-07-14,Peru,Romania,4,-2,Visitante
1494,1930-07-15,Argentina,France,1,1,Local


## 3. Resumen descriptivo

Estadísticas básicas (media, desviación, mínimos y máximos) de las variables numéricas
de marcador. Sirve para tener una primera intuición del rango típico de goles por partido.

In [4]:
print("--- RESUMEN DESCRIPTIVO ---")
display(procesador.resumen_descriptivo())

--- RESUMEN DESCRIPTIVO ---


,home_score,away_score,total_goles,diferencia_goles
count,1068.000000,1068.000000,1068.000000,1068.000000
mean,1.583333,1.251873,2.835206,0.331461
std,1.494000,1.292051,1.920880,2.028074
min,0.000000,0.000000,0.000000,-8.000000
25%,0.750000,0.000000,1.000000,-1.000000
50%,1.000000,1.000000,3.000000,0.000000
75%,2.000000,2.000000,4.000000,1.000000
max,10.000000,8.000000,12.000000,9.000000


**Interpretación:** el promedio de `total_goles` por partido ronda los ~2.9 goles, con
un máximo histórico muy por encima de la media. Esto ya sugiere que la distribución no
es simétrica: hay muchos partidos de pocos goles y unos pocos con marcadores muy altos,
hipótesis que confirmaremos con la asimetría más adelante.

## 4. Matriz de correlación

Relación lineal entre las variables numéricas de marcador. Es útil para verificar
dependencias esperadas y descartar redundancias.

In [5]:
print("--- MATRIZ DE CORRELACIÓN ---")
display(procesador.matriz_correlacion())

--- MATRIZ DE CORRELACIÓN ---


,home_score,away_score,total_goles,diferencia_goles
home_score,1.000000,-0.054823,0.740893,0.771586
away_score,-0.054823,1.000000,0.629995,-0.677469
total_goles,0.740893,0.629995,1.000000,0.144426
diferencia_goles,0.771586,-0.677469,0.144426,1.000000


**Interpretación:** `total_goles` correlaciona positivamente con `home_score` y
`away_score` por construcción (es su suma). La `diferencia_goles` se mueve con el marcador
local y en sentido inverso al visitante, lo cual es coherente con su definición.

## 5. Forma de la distribución de goles

Medimos la **asimetría** (*skewness*) y la **curtosis** de `total_goles` para describir
la forma de la distribución, más allá de la media y la desviación.

In [6]:
forma = procesador.analizar_asimetria_y_curtosis()
for clave, valor in forma.items():
    print(f"{clave}: {valor}")

Asimetría de Goles: 0.9378
Curtosis de Goles: 1.3992
Interpretación: Distribución asimétrica positiva (cola a la derecha), típica en deportes donde los marcadores abultados son poco frecuentes.


**Interpretación:** la asimetría es **positiva (~0.91)**, lo que confirma la *cola a la
derecha*: la mayoría de los partidos tienen pocos goles y existe una minoría de partidos
muy goleadores que estira la distribución. La curtosis (~1.31) indica una concentración
algo mayor que la de una distribución normal, con presencia de valores extremos.

## 6. Detección de partidos atípicos (regla del IQR)

Aplicamos la regla del **Rango Intercuartílico (IQR)** para identificar partidos cuyo
número total de goles es anormalmente alto (por encima de `Q3 + 1.5 · IQR`). Estos son
los *outliers* del dataset.

In [7]:
atipicos = procesador.detectar_partidos_atipicos_iqr()
display(atipicos[['date', 'home_team', 'away_team',
                  'home_score', 'away_score', 'total_goles']].head(10))

Identificados 11 partidos con marcadores atípicos (goles > 8.5).


,date,home_team,away_team,home_score,away_score,total_goles
3975,1954-06-26,Switzerland,Austria,5,7,12
3969,1954-06-20,Germany,Hungary,3,8,11
2329,1938-06-05,Brazil,Poland,6,5,11
13302,1982-06-15,Hungary,El Salvador,10,1,11
4700,1958-06-08,France,Paraguay,7,3,10
49518,2026-07-18,France,England,4,6,10
1500,1930-07-19,Argentina,Mexico,6,3,9
3962,1954-06-17,Hungary,South Korea,9,0,9
3972,1954-06-23,Germany,Turkey,7,2,9
9779,1974-06-18,Yugoslavia,DR Congo,9,0,9


**Interpretación:** el umbral de goles atípicos se sitúa en **8.5 goles por partido**, y
se detectan alrededor de **10 partidos** que lo superan. Son encuentros históricos con
marcadores excepcionalmente altos, que conviene tener presentes porque pueden distorsionar
los promedios generales.

## 7. Top de goleadas históricas

Finalmente, listamos los partidos con la **mayor diferencia de goles** de la historia de
los Mundiales, es decir, las victorias más contundentes.

In [8]:
top_goleadas = procesador.obtener_top_goleadas_historicas(top_n=10)
display(top_goleadas)

,date,home_team,away_team,home_score,away_score,total_goles
13302,1982-06-15,Hungary,El Salvador,10,1,11
9779,1974-06-18,Yugoslavia,DR Congo,9,0,9
3962,1954-06-17,Hungary,South Korea,9,0,9
26437,2002-06-01,Germany,Saudi Arabia,8,0,8
2340,1938-06-12,Cuba,Sweden,0,8,8
3405,1950-07-02,Bolivia,Uruguay,0,8,8
45729,2022-11-23,Spain,Costa Rica,7,0,7
33903,2010-06-21,Portugal,North Korea,7,0,7
9782,1974-06-19,Haiti,Poland,0,7,7
3970,1954-06-20,South Korea,Turkey,0,7,7


**Interpretación:** encabeza la lista la histórica goleada de **Hungría 10–1 a El
Salvador (1982)**, seguida de varios 9–0. Estos resultados coinciden en gran medida con
los outliers detectados por el IQR, lo que refuerza la validez de ambos métodos.

## Conclusiones del EDA

- La distribución de goles por partido está **sesgada a la derecha**: predominan los
  marcadores bajos con una minoría de partidos muy goleadores.
- La regla del **IQR** y el ranking de goleadas identifican de forma consistente los
  mismos partidos extremos, lo que da confianza en la limpieza de los datos.
- El dataset procesado queda listo en `data/processed/partidos-limpios.csv` para el
  notebook de **visualización**, donde exploraremos las historias detrás de estos números.